# Parcial Corte 2 - Cafetería U. Sabana

##  Abril 15 2026

### Mateo Perez, Simon Restrepo, Diego Limas 

In [65]:
import pandas as pd
import sqlite3

In [66]:
df_productos = pd.read_csv("parcial_2_productos_sucios.csv")
df_clientes = pd.read_csv("parcial_2_clientes_sucios.csv")
df_proveedores = pd.read_csv("parcial_2_proveedores_sucios.csv")

In [67]:
df_productos[["nombre_producto", "categoria"]] = df_productos["producto_categoria"].str.split(" - ", expand=True)

df_productos["precio"] = df_productos["precio"].replace(r"[\$,]", "", regex=True)
df_productos["precio"] = pd.to_numeric(df_productos["precio"], errors="coerce")

df_productos_limpio = df_productos.drop(columns=["producto_categoria"])

# 🔥 AGREGAR ID
df_productos_limpio["id_producto"] = range(1, len(df_productos_limpio) + 1)

df_productos_limpio = df_productos_limpio[
    ["id_producto", "nombre_producto", "categoria", "precio"]
]

In [68]:
df_clientes[["nombre_cliente", "tipo_cliente"]] = df_clientes["cliente_tipo"].str.split(" - ", expand=True)

df_clientes["nombre_cliente"] = df_clientes["nombre_cliente"].str.title()

# 🔥 CORRECCIÓN CLAVE (error anterior)
df_clientes["telefono"] = df_clientes["telefono"].fillna("Sin telefono").astype(str)

df_clientes_limpio = df_clientes.drop(columns=["cliente_tipo"])

# 🔥 AGREGAR ID
df_clientes_limpio["id_cliente"] = range(1, len(df_clientes_limpio) + 1)

df_clientes_limpio = df_clientes_limpio[
    ["id_cliente", "nombre_cliente", "tipo_cliente", "telefono"]
]

In [69]:
df_proveedores[["nombre_empresa", "ciudad"]] = df_proveedores["empresa_ciudad"].str.split(" - ", expand=True)

df_proveedores_limpio = df_proveedores.drop(columns=["empresa_ciudad"])

# 🔥 AGREGAR ID (soluciona tu error de JOIN)
df_proveedores_limpio["id_proveedor"] = range(1, len(df_proveedores_limpio) + 1)

df_proveedores_limpio = df_proveedores_limpio[
    ["id_proveedor", "nombre_empresa", "ciudad"]
]

In [70]:
df_productos_limpio.to_csv("parcial_2_productos_limpios.csv", index=False)
df_clientes_limpio.to_csv("parcial_2_clientes_limpios.csv", index=False)
df_proveedores_limpio.to_csv("parcial_2_proveedores_limpios.csv", index=False)

In [71]:
conexion = sqlite3.connect("parcial_2_cafeteria.db")
cursor = conexion.cursor()

cursor.execute("PRAGMA foreign_keys = ON;")

In [72]:
cursor.execute("DROP TABLE IF EXISTS ventas")
cursor.execute("DROP TABLE IF EXISTS clientes")
cursor.execute("DROP TABLE IF EXISTS productos")
cursor.execute("DROP TABLE IF EXISTS proveedores")

In [73]:
cursor.execute("""
CREATE TABLE productos (
    id_producto INTEGER PRIMARY KEY,
    nombre_producto TEXT,
    categoria TEXT,
    precio REAL
)
""")

cursor.execute("""
CREATE TABLE clientes (
    id_cliente INTEGER PRIMARY KEY,
    nombre_cliente TEXT,
    tipo_cliente TEXT,
    telefono TEXT NOT NULL
)
""")

cursor.execute("""
CREATE TABLE proveedores (
    id_proveedor INTEGER PRIMARY KEY,
    nombre_empresa TEXT,
    ciudad TEXT
)
""")

cursor.execute("""
CREATE TABLE ventas (
    id_venta INTEGER PRIMARY KEY AUTOINCREMENT,
    id_cliente INTEGER,
    id_producto INTEGER,
    id_proveedor INTEGER,
    cantidad INTEGER,
    FOREIGN KEY(id_cliente) REFERENCES clientes(id_cliente),
    FOREIGN KEY(id_producto) REFERENCES productos(id_producto),
    FOREIGN KEY(id_proveedor) REFERENCES proveedores(id_proveedor)
)
""")

conexion.commit()

In [74]:
df_productos_limpio.to_sql("productos", conexion, if_exists="append", index=False)
df_clientes_limpio.to_sql("clientes", conexion, if_exists="append", index=False)
df_proveedores_limpio.to_sql("proveedores", conexion, if_exists="append", index=False)

10

In [75]:
cursor.executemany("""
INSERT INTO ventas (id_cliente, id_producto, id_proveedor, cantidad)
VALUES (?, ?, ?, ?)
""", [
    (1,1,1,2),
    (2,2,2,1),
    (3,3,3,4),
    (1,2,1,3),
    (2,1,2,2)
])

conexion.commit()

In [76]:
consulta = """
SELECT v.id_venta, c.nombre_cliente, p.nombre_producto, pr.nombre_empresa, v.cantidad
FROM ventas v
JOIN clientes c ON v.id_cliente = c.id_cliente
JOIN productos p ON v.id_producto = p.id_producto
JOIN proveedores pr ON v.id_proveedor = pr.id_proveedor
"""

df_join = pd.read_sql_query(consulta, conexion)
df_join

,id_venta,nombre_cliente,nombre_producto,nombre_empresa,cantidad
0,1,Diego Zuluaga,Café Tostao,CoopCafe,2
1,2,Ana Gomez,chocolatina jet,Insumos Panaderos,1
2,3,Carlos Ruiz,Empanada,Lacteos El Prado,4
3,4,Diego Zuluaga,chocolatina jet,CoopCafe,3
4,5,Ana Gomez,Café Tostao,Insumos Panaderos,2


In [77]:
cursor.execute("""
UPDATE ventas
SET cantidad = 10
WHERE id_venta = 1
""")

conexion.commit()

In [78]:
cursor.execute("""
DELETE FROM ventas
WHERE id_venta = 3
""")

conexion.commit()

In [79]:
df_final = pd.read_sql_query(consulta, conexion)
df_final

,id_venta,nombre_cliente,nombre_producto,nombre_empresa,cantidad
0,1,Diego Zuluaga,Café Tostao,CoopCafe,10
1,2,Ana Gomez,chocolatina jet,Insumos Panaderos,1
2,4,Diego Zuluaga,chocolatina jet,CoopCafe,3
3,5,Ana Gomez,Café Tostao,Insumos Panaderos,2


In [80]:
conexion.close()